In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = ""

Wikipedia retriever

In [ ]:
from langchain_community.retrievers import WikipediaRetriever

In [ ]:
# initialize the retriever
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [ ]:
# define query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# get relevant wikipedia documents
docs = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(docs):
  print(f"\n---result {i + 1}---")
  print(f"content:\n{doc.page_content}...")

vector store retriever

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document


In [ ]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [ ]:
# initialize embedding model
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

# create chroma vector store in memory
vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection",
)

In [ ]:
# convert vector store into a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [ ]:
query = "what is chroma used for?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
  print(f"\n---result {i + 1}---")
  print(f"content:\n{doc.page_content}...")

MMR

In [ ]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [ ]:
from langchain_community.vectorstores import FAISS

embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model,
)

In [ ]:
from re import search
# enable MMR in the retriever
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5},
)

In [ ]:
query = "what is langchain"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
  print(f"\n---result {i + 1}---")
  print(f"content:\n{doc.page_content}...")

Multi-Query retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [ ]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [ ]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model,
)

In [ ]:
similarity_retrievers = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
    llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
)

In [ ]:
# query
query = "How to improve energy levels and maintain balance?"

In [ ]:
# retriever results
similarity_search = similarity_retrievers.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(similarity_search):
  print(f"\n---result {i + 1}---")
  print(f"content:\n{doc.page_content}...")

In [ ]:
for i, doc in enumerate(multiquery_results):
  print(f"\n---result {i + 1}---")
  print(f"content:\n{doc.page_content}...")

Contextual Compression Retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [ ]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model,
)

In [ ]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
# create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor,
)

In [ ]:
query = "what is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(compressed_results):
  print(f"\n---result {i + 1}---")
  print(f"content:\n{doc.page_content}...")